# FFT 与多项式乘法

在本部分中，你将实现 FFT，并将 FFT 作为一个黑箱应用于一些实际问题。

__请注意，你所编写的函数是相互依赖的。因此，要解决本笔记本中后续的问题，必须正确完成前面的题目。__

## 环境配置说明：
### 如果在本地运行：
#### 首次配置
* 安装Anaconda：访问 https://www.anaconda.com/products/distribution
* 创建conda环境：`conda create -n cs131232 python=3.8`
  * 详见 https://conda.io/projects/conda/en/latest/user-guide/tasks/manage-environments.html
* 激活环境：`conda activate cs131232`
* 安装pip：`conda install pip`
* 安装Jupyter：`conda install jupyter`
#### 每次启动
* 确保激活环境：`conda activate cs131232`
* 启动Jupyter：`jupyter notebook` 或 `jupyter lab`
* 运行以下单元格**并在需要时重启内核**

In [ ]:
# Install dependencies
!pip install -r requirements.txt --quiet

In [ ]:
import otter
import scipy

assert (
    otter.__version__ >= "4.4.1" and scipy.__version__ >= "1.10.0"
), "Please reinstall the requirements and restart your kernel."

grader = otter.Notebook("fft.ipynb")
import numpy as np
from numpy.random import randint
from time import time
import tqdm
import scipy
import matplotlib.pyplot as plt
import numpy.random as random

rng_seed = 0

### Q1.1) 单位根（10分）

$n$ 次单位根定义为满足 $z^n=1$ 的复数。  
另一个等价的定义是数 $e^{\frac{2\pi ik}{n}}$，其中 $k={0, 1, \cdots, n-1}$。  
首先，编写一个函数，给定 $n$，输出所有 $n$ 次单位根。注意 $n$ 不一定是偶数。

你也可以选择以 $a+bi$ 的形式计算，但并非必须。

*提示：*
1. 计算单位根有多种方法。你可能会发现以下常量和函数很有用。根据你选择如何完成此部分，你可能不需要全部使用它们：
    * `np.e`
    * `np.pi`
    * `np.exp()`
    * `np.zeros()`
    * `np.roots()` 
2. 如果你不确定某个函数的具体用法，可以随时查阅相应的 Numpy/Scipy/Python 文档，或者在笔记本单元格中输入 `?` 后跟函数名以获取更多信息。例如，`?np.exp` 会提供关于 `np.exp()` 函数的更多信息。

3. Python 支持复数运算。你可以使用 `a + bj` 的形式输入复数，其中 `a` 和 `b` 是整数。例如，可以使用 `2 + 3j`。如果要将实数转换为复数类型，需要写成 `1 + 0j` 或 `complex(1)` 等形式。

4. 确保按正确顺序返回单位根：若主根为 $\omega$，则你应返回 $[1, \omega, \omega^2, \omega^3, \dots]$。

In [ ]:
def roots_of_unities(n):
    """
    args:
        n:int = n describes which root of unity to return
    return:
        a list of n complex numbers containing the n-th roots of unity [w^0, w^1, w^2, ..., w^{n-1}]
    """
    # w = e^{2*pi*i/n} is the principal n-th root of unity, so the roots in
    # order are [w^0, w^1, ..., w^{n-1}].
    return np.exp(2j * np.pi * np.arange(n) / n)


为了确保你的辅助函数正确，我们可以将计算得到的值绘制在单位圆上。运行下面的单元格，并确保输出符合你的预期：

In [ ]:
N = 16  # feel free to change this value and observe what happens
roots = roots_of_unities(N)

# Plot
f, ax = plt.subplots()
f.set_figwidth(4)
f.set_figheight(4)
plt.scatter([r.real for r in roots], [r.imag for r in roots])
ax.spines["left"].set_position("zero")
ax.spines["right"].set_color("none")
ax.yaxis.tick_left()
ax.spines["bottom"].set_position("zero")
ax.spines["top"].set_color("none")
ax.set_xlim([-1.2, 1.2])
ax.set_ylim([-1.2, 1.2])
ax.set_xticks([-1, -0.5, 0.5, 1])
ax.set_yticks([-1, -0.5, 0.5, 1])
ax.xaxis.tick_bottom()

In [ ]:
grader.check("q1.1")

## 离散傅里叶变换
在课堂上，我们将多项式从系数表示转换为点值表示（在 $2^n$ 次单位根处求值）的过程定义为“离散傅里叶变换”。  
例如，对于多项式 $P(x)=1+x+x^2+x^3$，其点值表示为 $P(1)=4, P(i)=0, P(-1)=0, P(-i)=0$。

### Q1.2) 朴素 DFT（10分）
编写一个实现 DFT 的朴素算法。

*提示：*
1. 注意我们列出系数的顺序。
2. 你可能会发现以下函数很有用。


In [ ]:
# Utility function, returns next 2^n
hyperceil = lambda x: int(2 ** np.ceil(np.log2(x)))

In [ ]:
def dft_naive(coeffs):
    """
    args:
        coeffs:np.array = list of numbers representing the coefficients of a polynomial where coeffs[i]
                    is the coeffiecient of the term x^i
    return:
        List containing the results of evaluating the polynomial at the roots of unity. Can be either a list or np.array
        [P(w_0), P(w_1), P(w_2), ...]
    """
    coeffs = np.asarray(coeffs, dtype=complex)
    n = len(coeffs)
    roots = roots_of_unities(n)
    powers = np.arange(n)
    # P(w_j) = sum_k coeffs[k] * w_j^k, computed naively for every root (O(n^2)).
    return np.array([np.sum(coeffs * roots[j] ** powers) for j in range(n)])


In [ ]:
grader.check("q1.2")

### Q1.3) FFT（20分）

FFT 是一种在 $\mathcal{O}(n \log n)$ 时间内计算 DFT 的算法。

现在，你将自行实现 FFT。按照这里的定义，它接受多项式的系数作为输入，在 $n$ 次单位根上求值，并返回这些值的列表。例如，调用

$$
FFT([1, 2, 3, 0], \; [1, i, -1, -i])
$$

应当对多项式 $1 + 2x + 3x^2$ 在点 $1, i, -1, -i$ 上求值，返回

$$
[6, \; -2 + 2i, \; 2, \; -2 - 2i]
$$

回顾一下，为了高效地对多项式

$$
P(x) = a_0 + a_1x + a_2x^2 + \cdots + a_{n - 1} x^{n - 1}
$$

进行这一计算，我们定义另外两个多项式 $E$ 和 $O$，分别包含偶次项和奇次项的系数，

$$
E(x) := a_0 + a_2x + \cdots + a_{n - 2}x^{n/2 - 1}, \qquad O(x) := a_1 + a_3x + \cdots = a_{n - 1}x^{n/2 - 1}
$$

它们满足关系式

$$
P(x) = E(x^2) + xO(x^2)
$$

我们在 $E$ 和 $O$ 上递归运行 FFT，在 $n/2$ 次单位根上对它们求值，然后利用上述关系式，用这些值来计算 $P$ 在 $n$ 次单位根上的取值。

请按照下面的步骤实现该过程，其中 `coeffs` 是待求值多项式的系数（$x^i$ 的系数位于索引 $i$ 处），且

$$
\text{roots} = [1, \omega, \omega^2, \ldots, \omega^{n - 1}]
$$

其中 $\omega$ 是某个本原 $n$ 次单位根，且 $n$ 为 $2$ 的幂。（注意：Python 中复数的算术运算与浮点数或整数的运算方式相同。此外，你可以使用 `A[::k]` 来获取数组 `A` 中每隔 $k$ 个元素的值。）

*调试提示：*
1. 请记住，你的输入是 `np.array` 类型。除非你特别将其转换为 `List` 类型，否则不能像 Python 列表那样对 `np.array` 进行追加或连接操作。你可以使用 `np.append` 向数组追加元素，使用 `np.concatenate` 连接两个 `np.array`。

In [ ]:
def fft(coeffs, roots):
    """
    args:
        coeffs (np.array): array of numbers representing the coefficients of a polynomial where coeffs[i]
                    is the coeffiecient of the term x^i
        roots (np.array): array containing the roots of unity [w_0, w_1, w_2, ..., w_{n-1}]
    return:
        List containing the results of evaluating the polynomial at the roots of unity
        [P(w_0), P(w_1), P(w_2), ...]
    """
    n = len(coeffs)
    assert n == hyperceil(n)
    coeffs = np.asarray(coeffs, dtype=complex)
    roots = np.asarray(roots, dtype=complex)

    if n == 1:
        return coeffs

    # Split P into its even and odd parts: P(x) = E(x^2) + x * O(x^2).
    even = fft(coeffs[0::2], roots[0::2])
    odd = fft(coeffs[1::2], roots[0::2])

    # (w^k)^2 = w^{2k} = roots[2k], so the half-sized problem uses roots[::2].
    # For the top half of the roots, w^{k + n/2} = -w^k.
    half = n // 2
    return np.concatenate((even + roots[:half] * odd, even + roots[half:] * odd))


### 测试

这里提供一个简单的正确性检查来测试你的实现。调用 $FFT([1, 2, 3, 0], [1, 1j, -1, -1j])$ 应当输出 $[6, \; -2 + 2j, \; 2, \; -2 - 2j]$（Python 使用 $j$ 表示虚数单位，而不是 $i$）。

你也可以在这里随意添加自己的测试单元格！

In [ ]:
expected = [6, -2 + 2j, 2, -2 - 2j]
actual = fft([1, 2, 3, 0], [1, 1j, -1, -1j])
assert np.allclose(expected, actual), f"expected: {expected}\n actual: {actual}"
print("success")

如果你正确地实现了 FFT 算法，而不是在每个点上进行朴素求值，那么结果只应在 `roots` 参数为单位根时才能正确工作。因此，调用 `FFT([1, 2, 3, 0], [1, 2, 3, 4])` 不应当返回多项式 $1 + 2x + 3x^2$ 在输入点 $[1, 2, 3, 4]$ 上的值（这些值应为 $[6, 17, 34, 57]$）：


In [ ]:
not_expected = [6, 17, 34, 57]
actual = fft([1, 2, 3, 0], [1, 2, 3, 4])
assert not np.allclose(expected, actual), f"NOT expected: {expected}\n actual: {actual}"
print("success")

In [ ]:
grader.check("q1.3")

### Q1.4) 逆 FFT（10分）

既然你已经确认 FFT 的实现是正确的，接下来请实现 IFFT。

IFFT 的实现只需不到 3 行代码，你可以查阅 DPV 教材第 75 页末尾来了解如何完成。

*提示：*
1. Python 支持通过对复数取 $-1$ 次幂来计算其倒数。换句话说，假设 `x` 是一个复数，则其倒数为 `x**(-1)`。
2. 我们将 `roots` 作为参数传入是有原因的，请确保在你的 `fft` 和 `ifft` 函数中都使用该参数，而不是硬编码你的单位根。


In [ ]:
def ifft(vals, roots):
    """
    args:
        val (np.array): numpy array containing the results of evaluating the polynomial at the roots of unity
                    [P(w_0), P(w_1), P(w_2), ...]

        roots (np.array): numpy array containing the roots of unity [w_0, w_1, w_2, ..., w_{n-1}]

    return:
        List containing the results of evaluating the polynomial at the roots of unity. Can be a Python list
            or numpy array.
        [P(w_0), P(w_1), P(w_2), ...]
    """
    n = len(vals)
    assert n == hyperceil(n)
    roots = np.asarray(roots, dtype=complex)
    # The inverse DFT is the DFT with w replaced by w^{-1}, scaled by 1/n:
    # w^{-k} = 1 / w^k, so inverting the roots elementwise gives the reverse order.
    return fft(vals, 1 / roots) / n


In [ ]:
grader.check("q1.4")

## Q1.5) 多项式乘法（选做）

现在，你将使用 FFT 函数作为黑箱来实现多项式乘法。回顾一下，为了做到这一点，我们首先对每个多项式的系数运行 FFT，将它们在一个足够大的 2 的幂（记为 $n$）的 $n$ 次单位根上求值。然后我们将这些值逐点相乘，最后对这些值运行逆 FFT，以转换回系数形式，从而得到乘积的系数。要执行逆 FFT，我们可以简单地运行 FFT，但使用倒序的单位根，并在最后除以 $n$。

请注意，FFT 和 IFFT 只接受次数为 $2^n-1$ 的多项式，因此你需要将系数填充到下一个 `hyperceil(n)`。

你可能会发现定义 `pad` 函数很有帮助，但并非必须。


In [ ]:
def pad(coeffs, to):
    """
    args:
        coeffs:List[] = list of numbers representing the coefficients of a polynomial where coeffs[i]
                    is the coeffiecient of the term x^i
        to:int = the final length coeffs should be after padding

    return:
        List of coefficients zero padded to length 'to'
    """
    coeffs = np.asarray(coeffs, dtype=complex)
    if to <= len(coeffs):
        return coeffs[:to]
    return np.concatenate((coeffs, np.zeros(to - len(coeffs), dtype=complex)))


def poly_multiply(coeffs1, coeffs2):
    """
    args:
        coeffs1:List[] = list of numbers representing the coefficients of a polynomial where coeffs[i]
                    is the coeffiecient of the term x^i
        coeffs2:List[] = list of numbers representing the coefficients of a polynomial where coeffs[i]
                    is the coeffiecient of the term x^i

    return:
        List of coefficients corresponding to the product polynomial of the two inputs.
    """
    # The product of two polynomials of lengths n1, n2 has length n1 + n2 - 1,
    # so we evaluate both on n >= n1 + n2 - 1 points where n is a power of 2.
    n = hyperceil(len(coeffs1) + len(coeffs2) - 1)
    roots = roots_of_unities(n)
    vals1 = fft(pad(coeffs1, n), roots)
    vals2 = fft(pad(coeffs2, n), roots)
    # Multiply pointwise, then interpolate back to coefficient form.
    return ifft(vals1 * vals2, roots)


### 测试

In [ ]:
def round_complex_to_int(lst):
    return [round(x.real) for x in lst]


def zero_pop(lst):
    return np.trim_zeros(lst, "b")

以下是一些用于检查你的解决方案的正确性测试。

In [ ]:
expected = [4, 13, 22, 15]
actual = round_complex_to_int(poly_multiply([1, 2, 3], [4, 5]))
print("expected: {}".format(expected))
print("actual:   {}\n".format(actual))

expected = [4, 13, 28, 27, 18, 0, 0, 0]
actual = round_complex_to_int(poly_multiply([1, 2, 3], [4, 5, 6]))
print("expected: {}".format(expected))
print("actual:   {}".format(actual))

FFT 的一个特点是，我们使用复数来乘整数多项式，因此这会引入浮点误差。你可以通过下面的调用来观察到这一点，该调用很可能不会返回精确的整数值（除非你在实现中做了某些处理来应对这一问题）：

In [ ]:
result = poly_multiply([1, 2, 3], [4, 5, 6])
result

因此，如果我们只关心整数结果，就必须对结果进行四舍五入：

In [ ]:
result = round_complex_to_int(result)
result

然而，可能仍然需要去除末尾多余的零：

In [ ]:
zero_pop(result)

希望这恰好给出了我们通过普通多项式乘法会得到的结果：$[4, 13, 28, 27, 18]$。

### 运行时间比较

这里，我们将使用 FFT 的多项式乘法的运行时间与朴素算法进行比较。

In [ ]:
def poly_multiply_naive(coeffs1, coeffs2):
    n1, n2 = len(coeffs1), len(coeffs2)
    n = n1 + n2 - 1
    prod_coeffs = [0] * n
    for deg in range(n):
        for i in range(max(0, deg + 1 - n2), min(n1, deg + 1)):
            prod_coeffs[deg] += coeffs1[i] * coeffs2[deg - i]
    return prod_coeffs

In [ ]:
poly_multiply_naive([3, 4, 5], [7, 2, 7, 4])

运行下面的单元格，你应该会看到，在输入较小时，FFT 的表现与朴素算法相当甚至略差；但一旦输入足够大，FFT 的表现会明显优于朴素算法——当输入规模最大时，朴素算法所需的等待时间会充分体现这一点（出于某种原因，你可能需要运行两次下一个单元格才能看到绘图）：

In [ ]:
def rand_ints(lo, hi, length):
    ints = list(randint(lo, hi, length))
    ints = [int(x) for x in ints]
    return ints


def record(array, value, name):
    array.append(value)
    print("%s%f" % (name, value))


fft_times = []
naive_times = []
speed_ups = []

for i in range(5):
    n = 10**i
    print("\nsize: %d" % n)
    poly1 = rand_ints(1, 100, n)
    poly2 = rand_ints(1, 100, n)
    time1 = time()
    fft_res = poly_multiply(poly1, poly2)
    fft_res = zero_pop(round_complex_to_int(fft_res))
    time2 = time()
    fft_time = time2 - time1
    record(fft_times, fft_time, "FFT time:   ")
    naive_res = poly_multiply_naive(poly1, poly2)
    time3 = time()
    naive_time = time3 - time2
    record(naive_times, naive_time, "naive time: ")
    assert fft_res == naive_res
    speed_up = naive_time / fft_time
    record(speed_ups, speed_up, "speed up: ")

plt.plot(fft_times, label="FFT")
plt.plot(naive_times, label="Naive")
plt.xlabel("Log Input Size")
plt.ylabel("Run Time (seconds)")
plt.legend(loc="upper left")
plt.title("Polynomial Multiplication Runtime")

plt.figure()
plt.plot(speed_ups)
plt.xlabel("Log Input Size")
plt.ylabel("Speedup")
plt.title("FFT Polynomial Multiplication Speedup")

## 提交

确保在运行下面的单元格之前，你已经按顺序运行了笔记本中的所有单元格，以便所有图像/图表都能在输出中显示。下面的单元格将为你生成一个用于提交的压缩文件。

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)